<a href="https://colab.research.google.com/github/ramyaa-hegde/flyrank-ml-internship-tasks/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramyaa-hegde/flyrank-ml-internship-tasks/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### 1. Action Hierarchy & Reason Codes

We map raw model outputs (opportunity scores) and ranking performance into four explicit, operational reason codes:

1. **`OPTIMIZE_TITLE_CTR`**: Striking-distance pages (ranks 4.0–20.0) with high model opportunity scores ($\ge 0.60$). Target action: Rewrite metadata and test title tags to capture search intent.
2. **`REFRESH_CONTENT`**: Lower-ranking pages (ranks > 20.0) demonstrating strong historical engagement signals. Target action: Update outdated information, expand content depth, and add internal links.
3. **`PROTECT_BRAND_RANK`**: Top-performing pages (ranks < 4.0). Target action: Monitor ranking stability and avoid disruptive URL or core content shifts.
4. **`MONITOR`**: Low engagement and poor position pages. Target action: No immediate engineering or content resources allocated.

In [6]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier

# 1. Connect and query sample dataset from Hugging Face DuckDB Warehouse
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("DROP SECRET IF EXISTS hf_secret;")
con.execute(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

parquet_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

df = con.execute(f"""
    SELECT
        client_hash_id,
        COALESCE(scroll_events, 0) AS scroll_events,
        COALESCE(gsc_avg_position, 100.0) AS gsc_avg_position,
        COALESCE(sessions_social, 0) AS sessions_social,
        CASE WHEN COALESCE(sessions_paid, 0) > 0 THEN 1 ELSE 0 END AS target_conversion
    FROM read_parquet('{parquet_path}')
    USING SAMPLE 20000 ROWS
""").df()

# 2. Train baseline Random Forest Model
X = df[['scroll_events', 'gsc_avg_position', 'sessions_social']]
y = df['target_conversion']

clf = RandomForestClassifier(n_estimators=100, max_depth=8, class_weight='balanced', random_state=42)
clf.fit(X, y)

# Predict opportunity probabilities
df['opportunity_score'] = clf.predict_proba(X)[:, 1]

# 3. Assign Reason Codes
def assign_reason_code(row):
    pos = row['gsc_avg_position']
    score = row['opportunity_score']
    if 4.0 <= pos <= 20.0 and score >= 0.60:
        return 'OPTIMIZE_TITLE_CTR'
    elif pos > 20.0 and score >= 0.50:
        return 'REFRESH_CONTENT'
    elif pos < 4.0:
        return 'PROTECT_BRAND_RANK'
    else:
        return 'MONITOR'

df['reason_code'] = df.apply(assign_reason_code, axis=1)

print("=== Ranked Action Distribution ===")
print(df['reason_code'].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Ranked Action Distribution ===
reason_code
MONITOR               17998
PROTECT_BRAND_RANK     1990
OPTIMIZE_TITLE_CTR        7
REFRESH_CONTENT           5
Name: count, dtype: int64


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### 2. Intended Use and Operating Limits

* **Intended Users:** Content strategists, SEO managers, and growth engineers.
* **Intended Use:** Prioritizing weekly content update queues and identifying high-upside pages that require title/meta-tag revisions.
* **Operating Limits:**
  * **Domain Shift Limit:** Not valid for newly published pages (< 14 days old) without Search Console telemetry.
  * **Algorithmic Limit:** Model provides directional scoring based on historical March 2026 patterns; it does not guarantee instant Google re-indexing or rank increases.

In [7]:
# Verify limits check on thin/new pages lacking position history
thin_pages = df[df['gsc_avg_position'] == 100.0]
print(f"Total unindexed/thin pages correctly flagged for exclusion or low score: {len(thin_pages)}")

Total unindexed/thin pages correctly flagged for exclusion or low score: 12465


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### 3. Human Review Requirements & No-Go Rules

#### Human Review Rules
* Every recommendation in `OPTIMIZE_TITLE_CTR` must undergo human editor review before updating production site metadata.
* Legal and compliance landing pages must be manually audited to prevent removing required disclosures.

#### The No-Go List (Never Automate)
1. **Never Automate URL Slugs / Redirects:** Changing live URLs based on scores risks broken backlinks and 404 errors.
2. **Never Automate Core Brand Keywords:** High-converting brand keywords must not be overwritten by automated copy generators.
3. **Never Automate Deletions:** Pages flagged for refresh should never be automatically unpublished without human consent.

In [8]:
# Verify No-Go List Guardrail: Ensure top brand pages are protected
protected_pages = df[df['reason_code'] == 'PROTECT_BRAND_RANK']
print(f"✓ Guardrail Active: {len(protected_pages)} high-performing pages protected from automated content rewrites.")

✓ Guardrail Active: 1990 high-performing pages protected from automated content rewrites.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### 4. Monitoring & Model Retrain Triggers

The action playbook signals become stale when search dynamics shift. The model must be re-trained or audited under the following triggers:

1. **Performance Decay:** More than 25% of recommended `OPTIMIZE_TITLE_CTR` pages experience position drops over a 30-day window.
2. **Data Drift:** Significant changes in Google Search Console metric distributions (e.g., broad core algorithm updates).
3. **Cadence Trigger:** Retrain model monthly on the newest data release partition (e.g., streaming April 2026 warehouse data).

In [9]:
# Simulated Drift Trigger Metric: Mean Opportunity Score Tracking
mean_score = df['opportunity_score'].mean()
print(f"Current Baseline Mean Opportunity Score: {mean_score:.4f}")
print("Trigger Status: Drift monitoring operational.")

Current Baseline Mean Opportunity Score: 0.0330
Trigger Status: Drift monitoring operational.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### 5. Exports for the Research Paper & Capstone

We export the processed recommendation queue and summary action breakdown to `work/outputs/` for use in our capstone research paper.

In [10]:
import os

# Create outputs directory if it doesn't exist
os.makedirs('outputs', exist_ok=True)
os.makedirs('work/outputs', exist_ok=True)

# Export top 100 priority queue
export_df = df[df['reason_code'].isin(['OPTIMIZE_TITLE_CTR', 'REFRESH_CONTENT'])].sort_values(by='opportunity_score', ascending=False)

export_df.head(100).to_csv('work/outputs/refresh_queue_sample.csv', index=False)
export_df.head(100).to_csv('outputs/refresh_queue_sample.csv', index=False)

print("✓ Successfully exported priority action queue to work/outputs/refresh_queue_sample.csv")

✓ Successfully exported priority action queue to work/outputs/refresh_queue_sample.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.